In [ ]:
import requests
from bs4 import BeautifulSoup
import json
from urllib.parse import urljoin
BASE_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
def crawl_goods(category_name, url, max_pages=1):
    items = []
    for page in range(1, max_pages + 1):
        params = {"PageNumber": page}
        res = requests.get(url, headers=BASE_HEADERS, params=params)
        res.raise_for_status()
        soup = BeautifulSoup(res.text, "html.parser")
        products = soup.select("ul#yesNewList > li")
        if not products:
            break
        for li in products:
            img_el = li.select_one("img.lazy")
            if not img_el:
                continue
            title_el = li.select_one("div.info_row.info_name > a.gd_name")
            thumb_url = img_el.get("data-original") or img_el.get("src")
            price_el = li.select_one("div.info_row.info_price em.yes_b")
            if not title_el:
                continue
            title = title_el.get_text(strip=True)
            detail_url = urljoin("https://www.yes24.com", title_el.get("href", ""))
            # thumbnail = thumb_el.get("src") if thumb_el else None
            price_text = price_el.get_text(strip=True).replace(",", "") if price_el else ""
            try:
                price = int("".join(ch for ch in price_text if ch.isdigit()))
            except ValueError:
                price = None
            items.append({
                "category": category_name,
                "title": title,
                "price": price,
                "thumbnail": thumb_url,
                "detail_url": detail_url,
            })
    return items
goods = []
goods += crawl_goods("학습/독서", "https://www.yes24.com/product/category/display/006001083", max_pages=1)
goods += crawl_goods("디지털",   "https://www.yes24.com/product/category/display/006001089", max_pages=1)
goods += crawl_goods("디자인문구", "https://www.yes24.com/product/category/display/006001004", max_pages=1)
# len(goods), goods[:3]
print(goods)